In [11]:
from pathlib import Path
from collections import Counter
import csv
import math
import re
import sys
import zlib

# ============================================================
# CONFIGURAÇÃO
# ============================================================

ARQUIVO_FATORACOES = "apendice_fatoracoes_Rn_sem_PRP.tex"
ARQUIVO_PHIN10 = "Phin10.txt"

# O anexo possui n=1 + 857 índices não triviais
TOTAL_ESPERADO = 858

# Dez repdígitos Smith conhecidos, na forma (a,n)
SMITH_ESPERADOS = {
    (4, 1),
    (2, 2),
    (6, 3),
    (1, 4),
    (6, 7),
    (4, 10),
    (4, 20),
    (5, 27),
    (5, 32),
    (4, 55),
}

try:
    sys.set_int_max_str_digits(0)
except AttributeError:
    pass


# ============================================================
# FUNÇÕES ELEMENTARES
# ============================================================

def soma_digitos(n):
    return sum(int(c) for c in str(abs(int(n))))


def repunidade(n):
    return (10**n - 1) // 9


P_A = {
    1: 0,
    2: 2,
    3: 3,
    4: 4,
    5: 5,
    6: 5,
    7: 7,
    8: 6,
    9: 6,
}


# ============================================================
# REGEX DO ANEXO
# ============================================================

RE_LINHA = re.compile(
    r"^\s*(\d+)\s*&\s*(.*?)\\\\\s*$"
)

RE_FACTOR = re.compile(
    r"\\Rfactor\{(\d+)\}"
)

RE_FACTORPOW = re.compile(
    r"\\Rfactorpow\{(\d+)\}\{(\d+)\}"
)

RE_KCERT = re.compile(
    r"\\Kcert\{([0-9]+[LM]?)\}"
    r"\{(\d+)\}\{(\d+)\}\{(\d+)\}"
)


# ============================================================
# LEITURA DOS ARQUIVOS
# ============================================================

def verificar_arquivos():

    for nome in [ARQUIVO_FATORACOES, ARQUIVO_PHIN10]:

        if not Path(nome).exists():
            raise FileNotFoundError(
                f"Arquivo não encontrado: {nome}"
            )

    print("Arquivos encontrados:")
    print(" ", Path(ARQUIVO_FATORACOES).resolve())
    print(" ", Path(ARQUIVO_PHIN10).resolve())


def ler_phin10():

    banco = {}

    texto = Path(
        ARQUIVO_PHIN10
    ).read_text(
        encoding="utf-8",
        errors="ignore"
    )

    for linha in texto.splitlines():

        linha = linha.strip()

        if not linha:
            continue

        primeiro = linha.split()[0]

        if re.fullmatch(r"\d+[LM]?", primeiro):
            banco[primeiro] = linha

    return banco


# ============================================================
# LEITURA DA TABELA DE R_n
# ============================================================

def extrair_linhas_fatoracao(texto):

    linhas = []

    for linha in texto.splitlines():

        m = RE_LINHA.match(linha)

        if not m:
            continue

        n = int(m.group(1))
        expr = m.group(2)

        if (
            "\\Rfactor" in expr
            or "\\Kcert" in expr
            or (n == 1 and "\\(1\\)" in expr)
        ):
            linhas.append((n, expr))

    return linhas


def extrair_fatores_explicitos(expr):

    fatores = Counter()

    for p, e in RE_FACTORPOW.findall(expr):
        fatores[int(p)] += int(e)

    for p in RE_FACTOR.findall(expr):
        fatores[int(p)] += 1

    return fatores


def extrair_kcert(expr):

    return [
        (
            rotulo,
            int(ndig),
            int(prefixo),
            int(crc)
        )
        for rotulo, ndig, prefixo, crc
        in RE_KCERT.findall(expr)
    ]


# ============================================================
# FATORAÇÃO ELEMENTAR DO ÍNDICE n
# ============================================================

def fatoracao_indice(n):
    """
    Fatora apenas o índice n.
    Isto NÃO é teste de primalidade dos fatores do artigo.
    """

    fatores = {}
    x = n

    while x % 2 == 0:
        fatores[2] = fatores.get(2, 0) + 1
        x //= 2

    p = 3

    while p * p <= x:

        while x % p == 0:
            fatores[p] = fatores.get(p, 0) + 1
            x //= p

        p += 2

    if x > 1:
        fatores[x] = fatores.get(x, 0) + 1

    return fatores


def divisores(n):

    ds = [1]

    for p, e in fatoracao_indice(n).items():

        novos = []

        for d in ds:
            for k in range(e + 1):
                novos.append(d * p**k)

        ds = novos

    return ds


def mobius(n):

    fac = fatoracao_indice(n)

    if any(e > 1 for e in fac.values()):
        return 0

    return -1 if len(fac) % 2 else 1


# ============================================================
# Phi_n(10)
# ============================================================

_phi10_cache = {}


def phi10(n):
    """
    Calcula Phi_n(10) exatamente por

       Phi_n(10) =
       prod_{d|n} (10^d-1)^{mu(n/d)}.

    Não testa primalidade.
    """

    if n in _phi10_cache:
        return _phi10_cache[n]

    numerador = 1
    denominador = 1

    for d in divisores(n):

        mu = mobius(n // d)

        if mu == 1:
            numerador *= 10**d - 1

        elif mu == -1:
            denominador *= 10**d - 1

    q, r = divmod(
        numerador,
        denominador
    )

    if r != 0:
        raise ArithmeticError(
            f"Erro ao calcular Phi_{n}(10)."
        )

    _phi10_cache[n] = q

    return q


# ============================================================
# FATORES AURIFEUILLIANOS L/M
# ============================================================

def phi20_L_base(k):
    """
    Phi_20L(10^(2k+1)).
    """

    return (
        10**(8*k + 4)
        - 10**(7*k + 4)
        + 5 * 10**(6*k + 3)
        - 2 * 10**(5*k + 3)
        + 7 * 10**(4*k + 2)
        - 2 * 10**(3*k + 2)
        + 5 * 10**(2*k + 1)
        - 10**(k + 1)
        + 1
    )


def phi20_M_base(k):
    """
    Phi_20M(10^(2k+1)).
    """

    return (
        10**(8*k + 4)
        + 10**(7*k + 4)
        + 5 * 10**(6*k + 3)
        + 2 * 10**(5*k + 3)
        + 7 * 10**(4*k + 2)
        + 2 * 10**(3*k + 2)
        + 5 * 10**(2*k + 1)
        + 10**(k + 1)
        + 1
    )


_aurif_cache = {}


def componente_aurifeuilliano(n, lado):
    """
    Calcula Phi_{nL}(10) ou Phi_{nM}(10).

    Esses rótulos ocorrem para n = 40k+20.
    """

    chave = (n, lado)

    if chave in _aurif_cache:
        return _aurif_cache[chave]

    if n % 40 != 20:
        raise ValueError(
            f"{n}{lado}: índice não é 20 mod 40."
        )

    k = (n - 20) // 40

    Phi = phi10(n)

    if lado == "L":
        grande = phi20_L_base(k)

    elif lado == "M":
        grande = phi20_M_base(k)

    else:
        raise ValueError(
            f"Lado inválido: {lado}"
        )

    resultado = math.gcd(
        Phi,
        grande
    )

    _aurif_cache[chave] = resultado

    return resultado


# ============================================================
# CRC
# ============================================================

def crc32_decimal(n):

    return (
        zlib.crc32(
            str(n).encode("ascii")
        )
        & 0xffffffff
    )


# ============================================================
# IDENTIFICADOR KAMADA
# ============================================================

def verificar_identificador_no_phin10(
    rotulo,
    ndig,
    prefixo,
    crc,
    banco_phin10
):
    """
    Apenas verifica que o identificador do anexo
    realmente aparece na linha correspondente do Phin10.
    """

    if rotulo not in banco_phin10:

        raise RuntimeError(
            f"{rotulo}: linha não encontrada em Phin10.txt."
        )

    linha = banco_phin10[rotulo]

    identificador = (
        f"p{ndig}_{prefixo}_{crc}"
    )

    if identificador not in linha:

        raise RuntimeError(
            f"{rotulo}: identificador {identificador} "
            f"não aparece em Phin10.txt."
        )


# ============================================================
# COMPONENTE COMPLETO ASSOCIADO AO Kcert
# ============================================================

def valor_componente(rotulo):

    # Caso comum: d
    if rotulo.isdigit():
        return phi10(int(rotulo))

    # Caso Aurifeuilliano: dL ou dM
    m = re.fullmatch(
        r"(\d+)([LM])",
        rotulo
    )

    if not m:
        raise ValueError(
            f"Rótulo desconhecido: {rotulo}"
        )

    n = int(m.group(1))
    lado = m.group(2)

    return componente_aurifeuilliano(
        n,
        lado
    )


# ============================================================
# RECONSTRUIR UM Kcert
# ============================================================

def reconstruir_kcert(
    rotulo,
    ndig,
    prefixo,
    crc,
    fatores_explicitos,
    banco_phin10
):

    verificar_identificador_no_phin10(
        rotulo,
        ndig,
        prefixo,
        crc,
        banco_phin10
    )

    componente = valor_componente(
        rotulo
    )

    restante = componente

    # --------------------------------------------------------
    # Retirar do componente todos os fatores explícitos
    # que realmente o dividem
    # --------------------------------------------------------

    for p, expoente in fatores_explicitos.items():

        for _ in range(expoente):

            if restante % p == 0:
                restante //= p
            else:
                break

    candidato = restante

    s = str(candidato)

    # --------------------------------------------------------
    # Validar comprimento
    # --------------------------------------------------------

    if len(s) != ndig:

        raise RuntimeError(
            f"{rotulo}: fator reconstruído com "
            f"{len(s)} algarismos; esperado {ndig}."
        )

    # --------------------------------------------------------
    # Validar prefixo
    # --------------------------------------------------------

    if not s.startswith(str(prefixo)):

        raise RuntimeError(
            f"{rotulo}: prefixo incorreto.\n"
            f"Esperado: {prefixo}\n"
            f"Obtido:   {s[:10]}"
        )

    # --------------------------------------------------------
    # Validar CRC
    # --------------------------------------------------------

    crc_obtido = crc32_decimal(
        candidato
    )

    if crc_obtido != crc:

        raise RuntimeError(
            f"{rotulo}: CRC incorreto.\n"
            f"Esperado: {crc}\n"
            f"Obtido:   {crc_obtido}"
        )

    return candidato


# ============================================================
# VERIFICAÇÃO DE UMA LINHA
# ============================================================

def verificar_linha(
    n,
    expr,
    banco_phin10
):

    if n == 1:

        return {
            "n": 1,
            "Rn": 1,
            "produto_ok": True,
            "P_Rn": 0,
            "fatores": Counter(),
            "numero_kcert": 0,
        }

    Rn = repunidade(n)

    fatores_explicitos = (
        extrair_fatores_explicitos(expr)
    )

    fatores = Counter(
        fatores_explicitos
    )

    kcerts = extrair_kcert(expr)

    # --------------------------------------------------------
    # Reconstruir TODOS os fatores abreviados
    # --------------------------------------------------------

    for (
        rotulo,
        ndig,
        prefixo,
        crc
    ) in kcerts:

        candidato = reconstruir_kcert(
            rotulo,
            ndig,
            prefixo,
            crc,
            fatores_explicitos,
            banco_phin10
        )

        fatores[candidato] += 1

    # --------------------------------------------------------
    # Conferir produto
    # --------------------------------------------------------

    produto = 1

    for p, e in fatores.items():
        produto *= p**e

    if produto != Rn:

        if Rn % produto == 0:

            falta = Rn // produto

            raise RuntimeError(
                f"\nERRO EM n={n}\n"
                f"O produto é divisor de R_n, "
                f"mas falta um fator de "
                f"{len(str(falta))} algarismos."
            )

        raise RuntimeError(
            f"\nERRO EM n={n}\n"
            f"O produto dos fatores não é R_n."
        )

    # --------------------------------------------------------
    # Calcular P(R_n)
    # --------------------------------------------------------

    P_Rn = sum(
        e * soma_digitos(p)
        for p, e in fatores.items()
    )

    return {
        "n": n,
        "Rn": Rn,
        "produto_ok": True,
        "P_Rn": P_Rn,
        "fatores": fatores,
        "numero_kcert": len(kcerts),
    }


# ============================================================
# COMPOSTO SEM TESTE DE PRIMALIDADE
# ============================================================

def repunidade_composta_pela_fatoracao(
    resultado
):

    n = resultado["n"]

    if n == 1:
        return False

    fatores = resultado["fatores"]
    Rn = resultado["Rn"]

    # Um único fator igual ao próprio R_n
    # significa que a tabela o apresenta como primo.
    if len(fatores) == 1:

        p, e = next(
            iter(fatores.items())
        )

        if e == 1 and p == Rn:
            return False

    return True


# ============================================================
# TESTE DE SMITH
# ============================================================

def testar_smith(resultado):

    n = resultado["n"]
    P_Rn = resultado["P_Rn"]

    encontrados = []

    for a in range(1, 10):

        # ----------------------------------------------------
        # Smith exige composto
        # ----------------------------------------------------

        if n == 1:

            if a not in {4, 6, 8, 9}:
                continue

        elif a == 1:

            if not repunidade_composta_pela_fatoracao(
                resultado
            ):
                continue

        # n>=2 e a>=2:
        # a R_n é automaticamente composto.

        esquerda = (
            P_Rn + P_A[a]
        )

        direita = (
            a * n
        )

        if esquerda == direita:

            encontrados.append({
                "a": a,
                "n": n,
                "P_Rn": P_Rn,
                "P_a": P_A[a],
                "an": direita,
            })

    return encontrados


# ============================================================
# PROGRAMA PRINCIPAL
# ============================================================

def main():

    print("=" * 78)
    print("CHECAGEM DAS FATORAÇÕES DE R_n E DOS REPDÍGITOS SMITH")
    print("=" * 78)

    verificar_arquivos()

    # --------------------------------------------------------
    # Ler anexos
    # --------------------------------------------------------

    texto = Path(
        ARQUIVO_FATORACOES
    ).read_text(
        encoding="utf-8"
    )

    linhas = extrair_linhas_fatoracao(
        texto
    )

    print()
    print(
        "Linhas de fatoração encontradas:",
        len(linhas)
    )

    if len(linhas) != TOTAL_ESPERADO:

        raise RuntimeError(
            f"Esperava {TOTAL_ESPERADO} linhas, "
            f"mas encontrei {len(linhas)}."
        )

    banco_phin10 = ler_phin10()

    print(
        "Entradas lidas de Phin10.txt:",
        len(banco_phin10)
    )

    # --------------------------------------------------------
    # Executar
    # --------------------------------------------------------

    resultados = []
    smith = []

    print()
    print("Iniciando verificações...")

    for i, (n, expr) in enumerate(
        linhas,
        start=1
    ):

        resultado = verificar_linha(
            n,
            expr,
            banco_phin10
        )

        resultados.append(
            resultado
        )

        smith.extend(
            testar_smith(resultado)
        )

        if (
            i % 25 == 0
            or i == len(linhas)
        ):

            print(
                f"  {i:4d}/{len(linhas)} "
                f"(último n={n})"
            )

    # ========================================================
    # RESULTADO DAS MULTIPLICAÇÕES
    # ========================================================

    corretos = sum(
        r["produto_ok"]
        for r in resultados
    )

    print()
    print("=" * 78)
    print("MULTIPLICAÇÕES")
    print("=" * 78)

    print(
        f"Produtos corretos: "
        f"{corretos}/{len(resultados)}"
    )

    if corretos != TOTAL_ESPERADO:

        raise RuntimeError(
            "Nem todas as fatorações foram verificadas."
        )

    print(
        "OK: todas as fatorações reproduzem exatamente R_n."
    )

    # ========================================================
    # RESULTADOS SMITH
    # ========================================================

    smith = sorted(
        smith,
        key=lambda x: (
            x["n"],
            x["a"]
        )
    )

    pares = {
        (x["a"], x["n"])
        for x in smith
    }

    print()
    print("=" * 78)
    print("REPDÍGITOS SMITH")
    print("=" * 78)

    for x in smith:

        print(
            f"a={x['a']}, "
            f"n={x['n']}, "
            f"P(R_n)={x['P_Rn']}, "
            f"P(a)={x['P_a']}, "
            f"P(R_n)+P(a)="
            f"{x['P_Rn'] + x['P_a']} "
            f"= {x['an']}"
        )

    print()
    print(
        "Número de casos Smith:",
        len(pares)
    )

    # ========================================================
    # VALIDAR OS 10 CASOS
    # ========================================================

    faltantes = (
        SMITH_ESPERADOS - pares
    )

    extras = (
        pares - SMITH_ESPERADOS
    )

    if faltantes or extras:

        print()
        print("Faltantes:", sorted(faltantes))
        print("Extras:", sorted(extras))

        raise RuntimeError(
            "A lista de Smith não coincide com "
            "os dez casos esperados."
        )

    print(
        "OK: exatamente os dez casos Smith conhecidos."
    )

    # ========================================================
    # CASO n=1
    # ========================================================

    smith_n1 = [
        (x["a"], x["n"])
        for x in smith
        if x["n"] == 1
    ]

    if smith_n1 != [(4, 1)]:

        raise RuntimeError(
            f"Erro no caso n=1: {smith_n1}"
        )

    print(
        "OK: para n=1 somente 4 é Smith."
    )

    # ========================================================
    # CSV COMPLETO
    # ========================================================

    with open(
        "checagem_completa_repdigitos.csv",
        "w",
        newline="",
        encoding="utf-8"
    ) as f:

        w = csv.writer(f)

        w.writerow([
            "n",
            "produto_ok",
            "P_Rn",
            "numero_Kcert",
        ])

        for r in resultados:

            w.writerow([
                r["n"],
                r["produto_ok"],
                r["P_Rn"],
                r["numero_kcert"],
            ])

    # ========================================================
    # CSV SMITH
    # ========================================================

    with open(
        "smith_encontrados.csv",
        "w",
        newline="",
        encoding="utf-8"
    ) as f:

        w = csv.writer(f)

        w.writerow([
            "a",
            "n",
            "P_Rn",
            "P_a",
            "an",
        ])

        for x in smith:

            w.writerow([
                x["a"],
                x["n"],
                x["P_Rn"],
                x["P_a"],
                x["an"],
            ])

    # ========================================================
    # FINAL
    # ========================================================

    print()
    print("=" * 78)
    print("CHECAGEM CONCLUÍDA")
    print("=" * 78)

    print(
        f"{TOTAL_ESPERADO}/{TOTAL_ESPERADO} "
        "fatorações verificadas."
    )

    print("Produtos incorretos: 0.")
    print("Casos Smith encontrados: 10.")
    print("Novos casos Smith: 0.")

    print()
    print("Arquivos gerados:")
    print("  checagem_completa_repdigitos.csv")
    print("  smith_encontrados.csv")


# ============================================================
# EXECUTAR
# ============================================================

if __name__ == "__main__":
    main()

CHECAGEM DAS FATORAÇÕES DE R_n E DOS REPDÍGITOS SMITH
Arquivos encontrados:
  C:\Users\User\apendice_fatoracoes_Rn_sem_PRP.tex
  C:\Users\User\Phin10.txt

Linhas de fatoração encontradas: 858
Entradas lidas de Phin10.txt: 307500

Iniciando verificações...
    25/858 (último n=25)
    50/858 (último n=50)
    75/858 (último n=75)
   100/858 (último n=100)
   125/858 (último n=125)
   150/858 (último n=150)
   175/858 (último n=175)
   200/858 (último n=200)
   225/858 (último n=225)
   250/858 (último n=250)
   275/858 (último n=275)
   300/858 (último n=300)
   325/858 (último n=325)
   350/858 (último n=350)
   375/858 (último n=376)
   400/858 (último n=406)
   425/858 (último n=440)
   450/858 (último n=471)
   475/858 (último n=501)
   500/858 (último n=538)
   525/858 (último n=573)
   550/858 (último n=610)
   575/858 (último n=648)
   600/858 (último n=690)
   625/858 (último n=744)
   650/858 (último n=805)
   675/858 (último n=882)
   700/858 (último n=966)
   725/858 (último 